# ***Version load "ibody_all" from CSV with mapping Dict "rename_map"***

***Question***This is too good to be true. I have an InBody 770 and an InBody 970 that include the time, the person, and the account in the exported CSV file. It would be wonderful if I could put them all into a single massive data frame and then sort them in Jupyter instead of keeping track of all those separate data classes in their own directories. Is that what you just told me I could accomplish?

# ***Imports And Mapping Dictionaries***

## ***Import Libraries***

In [1]:
# Imports required for Loading, sorting .csx files to create specific data sets ie mrn inbody readings. 
%run ./sys_funcs.py              # loads all the def functions in sys_funcs.py into memory
#import sys_funcs                 # gives access to these def function digitalform that are in memory
from pathlib import Path
import csv
import pandas as pd
import numpy as np
# import tkinter as tk
import pickle
from pathlib import Path
import csv
import os
import sys
from datetime import datetime
from datetime import time
from sys_funcs import read_csv_to_array
from sys_funcs import clean_wsl_path
from sys_funcs import array_to_dt_row_dict
from sys_funcs import make_blnk_update_row_dict
from sys_funcs import transpose_csv_to_col_dict
#from sys_funcs import update_values_with_config, get_update_result
from sys_funcs import transfer_updates
from sys_funcs import get_dtv_range
from sys_funcs import universal_import
from sys_funcs import parse_inbody_timestamp
from sys_funcs import build_lut
from sys_funcs import extract_a_column_as_df
from sys_funcs import extract_multicolumns_as_df
from sys_funcs import validate_and_sort_timestamps
from sys_funcs import extract_and_filter_by_time_window
from sys_funcs import read_file_dual_path
from sys_funcs import write_file_dual_path
from sys_funcs import asc_to_csv_cnv
from collections.abc import Mapping
import re
#from sys_funcs import 

## ***ib97_columns =***  and ***mapping_770_970 =***

### 1. Define canonical IB970 schema
You must list every IB970 column here.
(Use your real list — I show the structure.)

In [2]:
ib97_columns = [
    "03 Person",
    "01 Date",
    "02 Time",
    "1. Name",
    "2. ID",
    "3. Height",
    "4. Date of Birth",
    "5. Gender",
    "6. Age",
    "7. Mobile Number",
    "8. Phone Number",
    "9. Zip Code",
    "10. Address",
    "11. E-mail",
    "12. Date of Registration",
    "13. Memo",
    "14. Test Date / Time",
    "15. Weight",
    "16. Lower Limit (Weight Normal Range)",
    "17. Upper Limit (Weight Normal Range)",
    "18. TBW (Total Body Water)",
    "19. Lower Limit (TBW Normal Range)",
    "20. Upper Limit (TBW Normal Range)",
    "21. ICW (Intracellular Water)",
    "22. Lower Limit (ICW Normal Range)",
    "23. Upper Limit (ICW Normal Range)",
    "24. ECW (Extracellular Water)",
    "25. Lower Limit (ECW Normal Range)",
    "26. Upper Limit (ECW Normal Range)",
    "27. Protein",
    "28. Lower Limit (Protein Normal Range)",
    "29. Upper Limit (Protein Normal Range)",
    "30. Minerals",
    "31. Lower Limit (Minerals Normal Range)",
    "32. Upper Limit (Minerals Normal Range)",
    "33. DLM (Dry Lean Mass)",
    "34. BFM (Body Fat Mass)",
    "35. Lower Limit (BFM Normal Range)",
    "36. Upper Limit (BFM Normal Range)",
    "37. FFM (Fat Free Mass)",
    "38. SMM (Skeletal Muscle Mass)",
    "39. Lower Limit (SMM Normal Range)",
    "40. Upper Limit (SMM Normal Range)",
    "41. BMI (Body Mass Index)",
    "42. Lower Limit (BMI Normal Range)",
    "43. Upper Limit (BMI Normal Range)",
    "44. PBF (Percent Body Fat)",
    "45. Lower Limit (PBF Normal Range)",
    "46. Upper Limit (PBF Normal Range)",
    "47. Lean Mass of Right Arm",
    "48. Lower Limit (Lean Mass of Right Arm Normal Range)",
    "49. Upper Limit (Lean Mass of Right Arm Normal Range)",
    "50. Lean Mass(%) of Right Arm",
    "51. Lean Mass of Left Arm",
    "52. Lower Limit (Lean Mass of Left Arm Normal Range)",
    "53. Upper Limit (Lean Mass of Left Arm Normal Range)",
    "54. Lean Mass(%) of Left Arm",
    "55. Lean Mass of Trunk",
    "56. Lower Limit (Lean Mass of Trunk Normal Range)",
    "57. Upper Limit (Lean Mass of Trunk Normal Range)",
    "58. Lean Mass(%) of Trunk",
    "59. Lean Mass of Right Leg",
    "60. Lower Limit (Lean Mass of Right Leg Normal Range)",
    "61. Upper Limit (Lean Mass of Right Leg Normal Range)",
    "62. Lean Mass(%) of Right Leg",
    "63. Lean Mass of Left Leg",
    "64. Lower Limit (Lean Mass of Left Leg Normal Range)",
    "65. Upper Limit (Lean Mass of Left Leg Normal Range)",
    "66. Lean Mass(%) of Left Leg",
    "67. Leg Lean Mass",
    "68. TBW of Right Arm",
    "69. Lower Limit (TBW of Right Arm Normal Range)",
    "70. Upper Limit (TBW of Right Arm Normal Range)",
    "71. TBW of Left Arm",
    "72. Lower Limit (TBW of Left Arm Normal Range)",
    "73. Upper Limit (TBW of Left Arm Normal Range)",
    "74. TBW of Trunk",
    "75. Lower Limit (TBW of Trunk Normal Range)",
    "76. Upper Limit (TBW of Trunk Normal Range)",
    "77. TBW of Right Leg",
    "78. Lower Limit (TBW of Right Leg Normal Range)",
    "79. Upper Limit (TBW of Right Leg Normal Range)",
    "80. TBW of Left Leg",
    "81. Lower Limit (TBW of Left Leg Normal Range)",
    "82. Upper Limit (TBW of Left Leg Normal Range)",
    "83. ICW of Right Arm",
    "84. Lower Limit (ICW of Right Arm Normal Range)",
    "85. Upper Limit (ICW of Right Arm Normal Range)",
    "86. ICW of Left Arm",
    "87. Lower Limit (ICW of Left Arm Normal Range)",
    "88. Upper Limit (ICW of Left Arm Normal Range)",
    "89. ICW of Trunk",
    "90. Lower Limit (ICW of Trunk Normal Range)",
    "91. Upper Limit (ICW of Trunk Normal Range)",
    "92. ICW of Right Leg",
    "93. Lower Limit (ICW of Right Leg Normal Range)",
    "94. Upper Limit (ICW of Right Leg Normal Range)",
    "95. ICW of Left Leg",
    "96. Lower Limit (ICW of Left Leg Normal Range)",
    "97. Upper Limit (ICW of Left Leg Normal Range)",
    "98. ECW of Right Arm",
    "99. Lower Limit (ECW of Right Arm Normal Range)",
    "100. Upper Limit (ECW of Right Arm Normal Range)",
    "101. ECW of Left Arm",
    "102. Lower Limit (ECW of Left Arm Normal Range)",
    "103. Upper Limit (ECW of Left Arm Normal Range)",
    "104. ECW of Trunk",
    "105. Lower Limit (ECW of Trunk Normal Range)",
    "106. Upper Limit (ECW of Trunk Normal Range)",
    "107. ECW of Right Leg",
    "108. Lower Limit (ECW of Right Leg Normal Range)",
    "109. Upper Limit (ECW of Right Leg Normal Range)",
    "110. ECW of Left Leg",
    "111. Lower Limit (ECW of Left Leg Normal Range)",
    "112. Upper Limit (ECW of Left Leg Normal Range)",
    "113. ECW/TBW",
    "114. ECW/TBW of Right Arm",
    "115. ECW/TBW of Left Arm",
    "116. ECW/TBW of Trunk",
    "117. ECW/TBW of Right Leg",
    "118. ECW/TBW of Left Leg",
    "119. BFM of Right Arm",
    "120. BFM% of Right Arm",
    "121. BFM of Left Arm",
    "122. BFM% of Left Arm",
    "123. BFM of Trunk",
    "124. BFM% of Trunk",
    "125. BFM of Right Leg",
    "126. BFM% of Right Leg",
    "127. BFM of Left Leg",
    "128. BFM% of Left Leg",
    "129. InBody Score",
    "130. BFM Control",
    "131. FFM Control",
    "132. BMR (Basal Metabolic Rate)",
    "133. VFL (Visceral Fat Level)",
    "134. VFA (Visceral Fat Area)",
    "135. BCM (Body Cell Mass)",
    "136. Lower Limit (BCM Normal Range)",
    "137. Upper Limit (BCM Normal Range)",
    "138. AC (Arm Circumference)",
    "139. BMC (Bone Mineral Content)",
    "140. Lower Limit (BMC Normal Range)",
    "141. Upper Limit (BMC Normal Range)",
    "142. TBW/FFM",
    "143. FFMI (Fat Free Mass Index)",
    "144. FMI (Fat Mass Index)",
    "145. 5kHz-RA Phase Angle",
    "146. 5kHz-LA Phase Angle",
    "147. 5kHz-TR Phase Angle",
    "148. 5kHz-RL Phase Angle",
    "149. 5kHz-LL Phase Angle",
    "150. 50kHz-RA Phase Angle",
    "151. 50kHz-LA Phase Angle",
    "152. 50kHz-TR Phase Angle",
    "153. 50kHz-RL Phase Angle",
    "154. 50kHz-LL Phase Angle",
    "155. 250kHz-RA Phase Angle",
    "156. 250kHz-LA Phase Angle",
    "157. 250kHz-TR Phase Angle",
    "158. 250kHz-RL Phase Angle",
    "159. 250kHz-LL Phase Angle",
    "160. 50kHz-Whole Body Phase Angle",
    "161. Systolic",
    "162. Diastolic",
    "163. Pulse",
    "164. Mean Artery Pressure",
    "165. Pulse Pressure",
    "166. Rate Pressure Product",
    "167. InBody Type",
    "168. Local ID",
    "169. ICW/FFM%",
    "170. SMI (Skeletal Muscle Index)",
    "171. Medical History",
    "172. Group",
    "173. Recommended Calorie Intake",
    "174. Lower Limit (BMR Normal Range)",
    "175. Upper Limit (BMR Normal Range)",
    "176. SMI(ASM/Ht²)_T score",
    "177. SMI(ASM/Ht²)_Z score",
    "178. Whole Body ECW/TBW_T score",
    "179. Whole Body ECW/TBW_Z score",
    "180. VFA_T score",
    "181. VFA_Z score",
    "182. BMI_T score",
    "183. BMI_Z score",
    "184. Subcutaneous Fat",
    "185. Arms/Legs Fat",
    "186. Lower Limit (Subcutaneous Fat of Abdomen Normal Range)",
    "187. Upper Limit (Subcutaneous Fat of Abdomen Normal Range)",
    "188. Visceral Fat",
    "189. Lower Limit (Visceral Fat of Abdomen Normal Range)",
    "190. Upper Limit (Visceral Fat of Abdomen Normal Range)",
    "191. Lower Limit (BFM of Arm & Leg Normal Range)",
    "192. Upper Limit (BFM of Arm & Leg Normal Range)",
    "193. Lower Limit (BFM of Trunk Normal Range)",
    "194. Upper Limit (BFM of Trunk Normal Range)",
    "195. Abdominal Fat",
    "196. V/S Ratio(Visceral Fat Area/Subcutaneous Fat Area ratio)",
    "197. SFA(Subcutaneous Fat Area)",
    "198. WHtR(Waist-Height Ratio)",
    "199. BAI(Body Adiposity Index)",
    "200. ABSI(A Body Shaped Index)",
    "201. Conicity Index",
    "202. LVR(Lean Mass-Visceral Fat Area Ratio)",
    "203. Upper Limit (WHtR Normal Range)",
    "204. Upper Limit (BAI Normal Range)",
    "205. Upper Limit (ABSI Normal Range)",
    "206. Upper Limit (Conicity Index Normal Range)",
    "207. Upper Limit (LVR Normal Range)",
    "208. 50khz-Ab Impedance",
    "209. 250khz-Ab Impedance",
    "210. WC(Waist Circumference)",
    "211. Angle-R",
    "212. Angle_L",
    "213. Lower Limit (Abdominal Fat Normal Range)",
    "214. Upper Limit (Abdominal Fat Normal Range)",
    "215. Lower Limit (FFM Normal Range)",
    "216. Upper Limit (FFM Normal Range)",
    "217. SMM/WT",
    "218. Systolic2",
    "219. Diastolic2",
    "220. Pulse2",
    "221. Mean Artery Pressure2",
    "222. Pulse Pressure2",
    "223. Rate Pressure Product2",
    "224. BFM% of Whole Body",
    "225. ECM/BCM",
    "226. TBW/WT",
    "227. Lean Mass(%) of Right Arm (Ideal Weight)",
    "228. Lean Mass(%) of Left Arm (Ideal Weight)",
    "229. Lean Mass(%) of Trunk (Ideal Weight)",
    "230. Lean Mass(%) of Right Leg (Ideal Weight)",
    "231. Lean Mass(%) of Left Leg (Ideal Weight)",
    "232. Impedance Check",
    "233. BCM_T score",
    "234. BCM_Z score",
    "235. FFMI_T score",
    "236. FFMI_Z score",
    "237. FMI_T score",
    "238. FMI_Z score",
    "239. PBF_T score",
    "240. PBF_Z score",
    "241. Weight_T score",
    "242. Weight_Z score",
    "243. 50kHz-Whole Body Phase Angle_T score",
    "244. 50kHz-Whole Body Phase Angle_Z score",
    "245. TBW/WT_T Score",
    "246. TBW/WT_Z Score",
    "247. SMI(SMM/Wt)_T score",
    "248. SMI(SMM/Wt)_Z score",
    "249. ECM/BCM_T Score",
    "250. ECM/BCM Z Score",
    "99. MachineID"
]



In [3]:
# verify ib97_columns

### 2. Load your existing mapping dictionary
Yo#u said you already have this — perfect.

In [4]:
mapping_770_970 = {
    "1. Name": "name",
    "2. ID": "id",
    "3. Height": "height",
    "4. Date of Birth": "dob",
    "5. Gender": "gender",
    "6. Age": "age",
    "7. Mobile Number": "mobile",
    "8. Phone Number": "phone",
    "9. Zip Code": "zip",
    "10. Address": "address",
    "11. E-mail": "email",
    "12. Date of Registration": "reg_date",
    "13. Memo": "memo",
    "14. Test Date / Time": "test_datetime",

    "15. Weight": "weight",
    "16. Lower Limit (Weight Normal Range)": "weight_lo",
    "17. Upper Limit (Weight Normal Range)": "weight_hi",

    "18. TBW (Total Body Water)": "tbw",
    "19. Lower Limit (TBW Normal Range)": "tbw_lo",
    "20. Upper Limit (TBW Normal Range)": "tbw_hi",

    "21. ICW (Intracellular Water)": "icw",
    "22. Lower Limit (ICW Normal Range)": "icw_lo",
    "23. Upper Limit (ICW Normal Range)": "icw_hi",

    "24. ECW (Extracellular Water)": "ecw",
    "25. Lower Limit (ECW Normal Range)": "ecw_lo",
    "26. Upper Limit (ECW Normal Range)": "ecw_hi",

    "27. Protein": "protein",
    "28. Lower Limit (Protein Normal Range)": "protein_lo",
    "29. Upper Limit (Protein Normal Range)": "protein_hi",

    "30. Minerals": "minerals",
    "31. Lower Limit (Minerals Normal Range)": "minerals_lo",
    "32. Upper Limit (Minerals Normal Range)": "minerals_hi",

    "33. DLM (Dry Lean Mass)": "dlm",
    "34. BFM (Body Fat Mass)": "bfm",
    "35. Lower Limit (BFM Normal Range)": "bfm_lo",
    "36. Upper Limit (BFM Normal Range)": "bfm_hi",

    "37. FFM (Fat Free Mass)": "ffm",
    "38. SMM (Skeletal Muscle Mass)": "smm",
    "39. Lower Limit (SMM Normal Range)": "smm_lo",
    "40. Upper Limit (SMM Normal Range)": "smm_hi",

    "41. BMI (Body Mass Index)": "bmi",
    "42. Lower Limit (BMI Normal Range)": "bmi_lo",
    "43. Upper Limit (BMI Normal Range)": "bmi_hi",

    "44. PBF (Percent Body Fat)": "pbf",
    "45. Lower Limit (PBF Normal Range)": "pbf_lo",
    "46. Upper Limit (PBF Normal Range)": "pbf_hi",

    "47. Lean Mass of Right Arm": "lm_ra",
    "48. Lower Limit (Lean Mass of Right Arm Normal Range)": "lm_ra_lo",
    "49. Upper Limit (Lean Mass of Right Arm Normal Range)": "lm_ra_hi",
    "50. Lean Mass(%) of Right Arm": "lm_ra_pct",

    "51. Lean Mass of Left Arm": "lm_la",
    "52. Lower Limit (Lean Mass of Left Arm Normal Range)": "lm_la_lo",
    "53. Upper Limit (Lean Mass of Left Arm Normal Range)": "lm_la_hi",
    "54. Lean Mass(%) of Left Arm": "lm_la_pct",

    "55. Lean Mass of Trunk": "lm_tr",
    "56. Lower Limit (Lean Mass of Trunk Normal Range)": "lm_tr_lo",
    "57. Upper Limit (Lean Mass of Trunk Normal Range)": "lm_tr_hi",
    "58. Lean Mass(%) of Trunk": "lm_tr_pct",

    "59. Lean Mass of Right Leg": "lm_rl",
    "60. Lower Limit (Lean Mass of Right Leg Normal Range)": "lm_rl_lo",
    "61. Upper Limit (Lean Mass of Right Leg Normal Range)": "lm_rl_hi",
    "62. Lean Mass(%) of Right Leg": "lm_rl_pct",

    "63. Lean Mass of Left Leg": "lm_ll",
    "64. Lower Limit (Lean Mass of Left Leg Normal Range)": "lm_ll_lo",
    "65. Upper Limit (Lean Mass of Left Leg Normal Range)": "lm_ll_hi",
    "66. Lean Mass(%) of Left Leg": "lm_ll_pct",

    "67. Leg Lean Mass": "lm_leg",

    "68. TBW of Right Arm": "tbw_ra",
    "69. Lower Limit (TBW of Right Arm Normal Range)": "tbw_ra_lo",
    "70. Upper Limit (TBW of Right Arm Normal Range)": "tbw_ra_hi",

    "71. TBW of Left Arm": "tbw_la",
    "72. Lower Limit (TBW of Left Arm Normal Range)": "tbw_la_lo",
    "73. Upper Limit (TBW of Left Arm Normal Range)": "tbw_la_hi",

    "74. TBW of Trunk": "tbw_tr",
    "75. Lower Limit (TBW of Trunk Normal Range)": "tbw_tr_lo",
    "76. Upper Limit (TBW of Trunk Normal Range)": "tbw_tr_hi",

    "77. TBW of Right Leg": "tbw_rl",
    "78. Lower Limit (TBW of Right Leg Normal Range)": "tbw_rl_lo",
    "79. Upper Limit (TBW of Right Leg Normal Range)": "tbw_rl_hi",

    "80. TBW of Left Leg": "tbw_ll",
    "81. Lower Limit (TBW of Left Leg Normal Range)": "tbw_ll_lo",
    "82. Upper Limit (TBW of Left Leg Normal Range)": "tbw_ll_hi",

    "83. ICW of Right Arm": "icw_ra",
    "84. Lower Limit (ICW of Right Arm Normal Range)": "icw_ra_lo",
    "85. Upper Limit (ICW of Right Arm Normal Range)": "icw_ra_hi",

    "86. ICW of Left Arm": "icw_la",
    "87. Lower Limit (ICW of Left Arm Normal Range)": "icw_la_lo",
    "88. Upper Limit (ICW of Left Arm Normal Range)": "icw_la_hi",

    "89. ICW of Trunk": "icw_tr",
    "90. Lower Limit (ICW of Trunk Normal Range)": "icw_tr_lo",
    "91. Upper Limit (ICW of Trunk Normal Range)": "icw_tr_hi",

    "92. ICW of Right Leg": "icw_rl",
    "93. Lower Limit (ICW of Right Leg Normal Range)": "icw_rl_lo",
    "94. Upper Limit (ICW of Right Leg Normal Range)": "icw_rl_hi",

    "95. ICW of Left Leg": "icw_ll",
    "96. Lower Limit (ICW of Left Leg Normal Range)": "icw_ll_lo",
    "97. Upper Limit (ICW of Left Leg Normal Range)": "icw_ll_hi",

    "98. ECW of Right Arm": "ecw_ra",
    "99. Lower Limit (ECW of Right Arm Normal Range)": "ecw_ra_lo",
    "100. Upper Limit (ECW of Right Arm Normal Range)": "ecw_ra_hi",

    "101. ECW of Left Arm": "ecw_la",
    "102. Lower Limit (ECW of Left Arm Normal Range)": "ecw_la_lo",
    "103. Upper Limit (ECW of Left Arm Normal Range)": "ecw_la_hi",

    "104. ECW of Trunk": "ecw_tr",
    "105. Lower Limit (ECW of Trunk Normal Range)": "ecw_tr_lo",
    "106. Upper Limit (ECW of Trunk Normal Range)": "ecw_tr_hi",

    "107. ECW of Right Leg": "ecw_rl",
    "108. Lower Limit (ECW of Right Leg Normal Range)": "ecw_rl_lo",
    "109. Upper Limit (ECW of Right Leg Normal Range)": "ecw_rl_hi",

    "110. ECW of Left Leg": "ecw_ll",
    "111. Lower Limit (ECW of Left Leg Normal Range)": "ecw_ll_lo",
    "112. Upper Limit (ECW of Left Leg Normal Range)": "ecw_ll_hi",

    "113. ECW/TBW": "ecw_tbw",
    "114. ECW/TBW of Right Arm": "ecw_tbw_ra",
    "115. ECW/TBW of Left Arm": "ecw_tbw_la",
    "116. ECW/TBW of Trunk": "ecw_tbw_tr",
    "117. ECW/TBW of Right Leg": "ecw_tbw_rl",
    "118. ECW/TBW of Left Leg": "ecw_tbw_ll",

    "119. BFM of Right Arm": "bfm_ra",
    "120. BFM% of Right Arm": "bfm_ra_pct",
}


In [5]:
# verify mapping_770_970

# ✅ Self‑Contained InBody Importer + Ingest Pipeline
Everything below is drop‑in ready.

## 🧩 1. Importer that reads each CSV correctly

In [6]:
import os
import pandas as pd
import re

def import_inbody_folder(folder_path):
    """
    Reads each InBody CSV file as raw text.
    Extracts:
        line 0 → labels
        line 1 → values
    Handles malformed CSVs, mismatched column counts,
    stray commas, quotes, and extra delimiters.
    Returns list of (filename, df_single_row).
    """

    dfs = []

    for fname in os.listdir(folder_path):
        if not fname.lower().endswith(".csv"):
            continue

        full_path = os.path.join(folder_path, fname)

        # Read raw text lines
        with open(full_path, "r", encoding="latin1") as f:
            lines = f.read().splitlines()

        if len(lines) < 2:
            print(f"Skipping {fname}: not enough lines")
            continue

        raw_header = lines[0]
        raw_values = lines[1]

        # Split using a robust regex that handles:
        # - commas inside quotes
        # - stray commas
        # - inconsistent quoting
        splitter = re.compile(r''',(?=(?:[^"]*"[^"]*")*[^"]*$)''')

        header = splitter.split(raw_header)
        values = splitter.split(raw_values)

        # If values have extra columns, pad header
        if len(values) > len(header):
            header += [f"Extra_{i}" for i in range(len(values) - len(header))]

        # If header has extra columns, pad values
        if len(header) > len(values):
            values += [None] * (len(header) - len(values))

        df = pd.DataFrame([values], columns=header)

        dfs.append((fname, df))

    return dfs


### This importer is purpose‑built for InBody CSVs and does not depend on any other function.

In [7]:
def add_control_columns(df, filename):
    df["source_file"] = filename
    df["ingest_timestamp"] = pd.Timestamp.now()
    return df


In [8]:
def ingest_inbody(df_raw, ib97_columns, mapping_770_970):
    # 1. Rename 770 columns → 970 schema
    df = df_raw.rename(columns=mapping_770_970)

    # 2. Build missing columns in one shot (no fragmentation)
    missing_cols = [col for col in ib97_columns if col not in df.columns]
    if missing_cols:
        filler = pd.DataFrame([[None] * len(missing_cols)], columns=missing_cols)
        df = pd.concat([df, filler], axis=1)

    # 3. Reorder to canonical schema
    df = df[ib97_columns]

    return df


## 🧩 2. Your harmonizer (slightly improved)

In [9]:
folder = "/home/ratlabs/JL_2/data/inbody_all"

dfs_raw = import_inbody_folder(folder)

dfs_norm = []

for filename, df_raw_single in dfs_raw:
    df_norm = ingest_inbody(df_raw_single, ib97_columns, mapping_770_970)
    df_norm = add_control_columns(df_norm, filename)
    dfs_norm.append(df_norm)

df_unified = pd.concat(dfs_norm, ignore_index=True)


In [10]:
#df_unified.iloc[:,78:]

## 🧩 3. Add control columns

In [11]:
def add_control_columns(df, filename):
    df["source_file"] = filename
    df["ingest_timestamp"] = pd.Timestamp.now()
    return df


## 🚀 4. Unified ingestion pipeline (final)

In [17]:
df_unified

,03 Person,01 Date,02 Time,1. Name,2. ID,3. Height,4. Date of Birth,5. Gender,6. Age,7. Mobile Number,...,244. 50kHz-Whole Body Phase Angle_Z score,245. TBW/WT_T Score,246. TBW/WT_Z Score,247. SMI(SMM/Wt)_T score,248. SMI(SMM/Wt)_Z score,249. ECM/BCM_T Score,250. ECM/BCM Z Score,99. MachineID,source_file,ingest_timestamp
0,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,091725-1_20260928075102.csv,2026-09-29 02:10:44.762868
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,091725_20251215233324.csv,2026-09-29 02:10:44.766785
2,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,091725-1_20260927074322.csv,2026-09-29 02:10:44.770915
3,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,091725-1_20260927224528.csv,2026-09-29 02:10:44.774744


# end

## 🧠 4. The correct ingestion function
This function takes any InBody file (770 or 970) and outputs a dataframe in the IB970 schema.

In [8]:
def ingest_inbody(df_raw, ib97_columns, mapping_770_970):
    # 1. Standardize column names
    df = df_raw.rename(columns=mapping_770_970)

    # 2. Add missing columns
    for col in ib97_columns:
        if col not in df.columns:
            df[col] = None

    # 3. Reorder to canonical schema
    df = df[ib97_columns]

    return df


##   🔧 5. Add control columns (your workflow)
You already use filename‑based metadata — here is the correct version:

In [19]:
def add_control_columns(df, filename):
    df["source_file"] = filename
    df["ingest_timestamp"] = pd.Timestamp.now()
    return df


## 🚀 6. The unified pipeline (final)
This is the exact pipeline you should run:

In [25]:
dfs_norm = []

for df_raw_single in df_raw:
    df_norm = ingest_inbody(df_raw_single, ib97_columns, mapping_770_970)
    df_norm = add_control_columns(df_norm, "inbody_all")
    dfs_norm.append(df_norm)

df_unified = pd.concat(dfs_norm, ignore_index=True)



AttributeError: 'str' object has no attribute 'rename'

# done with new

## 3. Universal CSV importer

In [5]:
import os
import glob
import pandas as pd


def import_csv_folder(folder_path, pattern="*.csv"):
    """
    Replacement importer:
    Delegates directly to universal_import(), which already handles
    file discovery, reading, and DataFrame creation.
    """
    return universal_import(
        folder_path=folder_path,
        pattern=pattern
    )


## 4. Machine detection
IB970 has columns IB770 never has — use one of them.

In [7]:
def detect_machine(df):
    return "IB970" if "ECW Ratio" in df.columns else "IB770"


## 5. Harmonizer: convert ANY DF → IB970 schema
This is the heart of the system.

In [8]:
def ingest_inbody(df_raw, ib97_columns, rename_map):
    machine = detect_machine(df_raw)
    df = df_raw.copy()
    print(machine)
         
    # Apply your full mapping dictionary
    if machine == "IB770":
        df = df.rename(columns=rename_map)

    # Add missing IB970 columns
    for col in ib97_columns:
        if col not in df.columns:
            df[col] = None

    # Add machine ID
    df["MachineID"] = machine

    # Reorder to canonical schema
    return df[ib97_columns]


## 6. Add control columns (if needed)
1. Add this function: extract date/time from filename
2. Add this function to append control cp;s Before data

In [9]:
import re
from datetime import datetime
import os

def extract_datetime_from_filename(filename):
    """
    Extracts datetime from filename using multiple patterns.
    Returns (date_str, time_str) or (None, None) if no match.
    """

    base = os.path.basename(filename)

    # Pattern 1: 14-digit timestamp YYYYMMDDHHMMSS
    m = re.search(r"(\d{14})", base)
    if m:
        ts = m.group(1)
        dt = datetime.strptime(ts, "%Y%m%d%H%M%S")
        return dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M:%S")

    # Pattern 2: YYYY-MM-DD_HH-MM-SS
    m = re.search(r"(\d{4}-\d{2}-\d{2})[_-](\d{2}-\d{2}-\d{2})", base)
    if m:
        date_part = m.group(1)
        time_part = m.group(2).replace("-", ":")
        return date_part, time_part

    # Pattern 3: YYYY_MM_DD_HHMMSS
    m = re.search(r"(\d{4}_\d{2}_\d{2})[_-](\d{6})", base)
    if m:
        date_part = m.group(1).replace("_", "-")
        time_raw = m.group(2)
        time_part = f"{time_raw[0:2]}:{time_raw[2:4]}:{time_raw[4:6]}"
        return date_part, time_part

    # No match
    return None, None


In [ ]:
def add_control_columns(df, filename):
    # Extract from filename if missing
    date_from_file, time_from_file = extract_datetime_from_filename(filename)

    if "Date" not in df.columns or df["Date"].isna().all():
        df["Date"] = date_from_file

    if "Time" not in df.columns or df["Time"].isna().all():
        df["Time"] = time_from_file

    # Ensure Person and Account exist (still None unless you add logic)
    for col in ["Person", "Account"]:
        if col not in df.columns:
            df[col] = None

    return df


## 7. Full ingestion pipeline
Reads all CSVs, normalizes them, and produces one unified DataFrame.

In [ ]:
# loads the new from the inbody_all
df_raw = universal_import(
    folder_path="/home/ratlabs/JL_2/data/inbody_all",
    pattern="*"
)
df_raw

In [7]:
#folder_path = "/home/ratlabs/JL_2/data/inbody_all"
folder_path = "/home/ratlabs/JL_2/data/ib97"
folder_path = "/home/ratlabs/JL_2/data/ib97"

# dfs_raw = import_csv_folder(folder_path, pattern="*.csv")


# dfs_raw = import_csv_folder(folder_path)

# dfs_norm = []

#files = glob.glob(os.path.join(folder_path, "*.csv"))
dfs_raw = []
for f in files:
    #df_raw = pd.read_csv(f, encoding="cp1252")  # or your fallback loader
    dfs_raw = import_csv_folder(folder_path, pattern="*.csv")

    #df_norm = ingest_inbody(df_raw, ib97_columns, rename_map)
    #df_norm = add_control_columns(df_norm, f)
    #dfs_norm.append(df_norm)

# df_unified = pd.concat(dfs_norm, ignore_index=True)
# df_sorted = df_unified.sort_values(["Person", "Date", "Time"])


NameError: name 'files' is not defined

In [8]:
dfs_raw

[]

## 🎯 Final Output
***df_sorted*** is your complete unified ***IB970 DataFrame***, containing:

All IB970 readings

All IB770 readings mapped into IB970 schema

All control columns

MachineID

Sorted by Person → Date → Time

This is the exact architecture used in multi‑device clinical ingestion systems.

In [14]:
df_sorted

1. Name2. ID3. Height4. Date of Birth5. Gender6. Age7. Mobile Number8. Phone Number9. Zip Code10. Address11. E-mail12. Date of Registration13. Memo14. Test Date / Time15. Weight16. Lower Limit (Weight Normal Range)17. Upper Limit (Weight Normal Range)18. TBW (Total Body Water)19. Lower Limit (TBW Normal Range)20. Upper Limit (TBW Normal Range)21. ICW (Intracellular Water)22. Lower Limit (ICW Normal Range)23. Upper Limit (ICW Normal Range)24. ECW (Extracellular Water)25. Lower Limit (ECW Normal Range)26. Upper Limit (ECW Normal Range)27. Protein28. Lower Limit (Protein Normal Range)29. Upper Limit (Protein Normal Range)30. Minerals31. Lower Limit (Minerals Normal Range)32. Upper Limit (Minerals Normal Range)33. DLM (Dry Lean Mass)34. BFM (Body Fat Mass)35. Lower Limit (BFM Normal Range)36. Upper Limit (BFM Normal Range)37. FFM (Fat Free Mass)38. SMM (Skeletal Muscle Mass)39. Lower Limit (SMM Normal Range)40. Upper Limit (SMM Normal Range)41. BMI (Body Mass Index)42. Lower Limit (BMI Normal Range)43. Upper Limit (BMI Normal Range)44. PBF (Percent Body Fat)45. Lower Limit (PBF Normal Range)46. Upper Limit (PBF Normal Range)47. Lean Mass of Right Arm48. Lower Limit (Lean Mass of Right Arm Normal Range)49. Upper Limit (Lean Mass of Right Arm Normal Range)50. Lean Mass(%) of Right Arm51. Lean Mass of Left Arm52. Lower Limit (Lean Mass of Left Arm Normal Range)53. Upper Limit (Lean Mass of Left Arm Normal Range)54. Lean Mass(%) of Left Arm55. Lean Mass of Trunk56. Lower Limit (Lean Mass of Trunk Normal Range)57. Upper Limit (Lean Mass of Trunk Normal Range)58. Lean Mass(%) of Trunk59. Lean Mass of Right Leg60. Lower Limit (Lean Mass of Right Leg Normal Range)61. Upper Limit (Lean Mass of Right Leg Normal Range)62. Lean Mass(%) of Right Leg63. Lean Mass of Left Leg64. Lower Limit (Lean Mass of Left Leg Normal Range)65. Upper Limit (Lean Mass of Left Leg Normal Range)66. Lean Mass(%) of Left Leg67. Leg Lean Mass68. TBW of Right Arm69. Lower Limit (TBW of Right Arm Normal Range)70. Upper Limit (TBW of Right Arm Normal Range)71. TBW of Left Arm72. Lower Limit (TBW of Left Arm Normal Range)73. Upper Limit (TBW of Left Arm Normal Range)74. TBW of Trunk75. Lower Limit (TBW of Trunk Normal Range)76. Upper Limit (TBW of Trunk Normal Range)77. TBW of Right Leg78. Lower Limit (TBW of Right Leg Normal Range)79. Upper Limit (TBW of Right Leg Normal Range)80. TBW of Left Leg81. Lower Limit (TBW of Left Leg Normal Range)82. Upper Limit (TBW of Left Leg Normal Range)83. ICW of Right Arm84. Lower Limit (ICW of Right Arm Normal Range)85. Upper Limit (ICW of Right Arm Normal Range)86. ICW of Left Arm87. Lower Limit (ICW of Left Arm Normal Range)88. Upper Limit (ICW of Left Arm Normal Range)89. ICW of Trunk90. Lower Limit (ICW of Trunk Normal Range)91. Upper Limit (ICW of Trunk Normal Range)92. ICW of Right Leg93. Lower Limit (ICW of Right Leg Normal Range)94. Upper Limit (ICW of Right Leg Normal Range)95. ICW of Left Leg96. Lower Limit (ICW of Left Leg Normal Range)97. Upper Limit (ICW of Left Leg Normal Range)98. ECW of Right Arm99. Lower Limit (ECW of Right Arm Normal Range)100. Upper Limit (ECW of Right Arm Normal Range)101. ECW of Left Arm102. Lower Limit (ECW of Left Arm Normal Range)103. Upper Limit (ECW of Left Arm Normal Range)104. ECW of Trunk105. Lower Limit (ECW of Trunk Normal Range)106. Upper Limit (ECW of Trunk Normal Range)107. ECW of Right Leg108. Lower Limit (ECW of Right Leg Normal Range)109. Upper Limit (ECW of Right Leg Normal Range)110. ECW of Left Leg111. Lower Limit (ECW of Left Leg Normal Range)112. Upper Limit (ECW of Left Leg Normal Range)113. ECW/TBW114. ECW/TBW of Right Arm115. ECW/TBW of Left Arm116. ECW/TBW of Trunk117. ECW/TBW of Right Leg118. ECW/TBW of Left Leg119. BFM of Right Arm120. BFM% of Right Arm121. BFM of Left Arm122. BFM% of Left Arm123. BFM of Trunk124. BFM% of Trunk125. BFM of Right Leg126. BFM% of Right Leg127. BFM of Left Leg128. BFM% of Left Leg129. InBody Score130. BFM Control131. FFM Control132. BMR (Ba

In [ ]:
# end?????===================